# Evaluation Script

Remember to run `scis2026_1_create_sample.ipynb` first, then this file.

## Initializations

### Imports

In [ ]:
import os
import time

import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.preprocessing import OneHotEncoder

from k_anonymization import datasets
from k_anonymization.algorithms import *
from k_anonymization.evaluation import metrics
from k_anonymization.evaluation.utils import get_equivalence_qids

### Constants

Use a seed for reproducibility

In [ ]:
SEED = 202512
SAMPLE_FRAC = 0.1

DATASET = datasets.ADULT

K = [2] + list(range(5, 101, 5))

In [ ]:
BASE_PATH = f"./results/{DATASET.name.replace(" ", "_")}"
_sample = pd.read_csv(f"{BASE_PATH}/sample/data.csv")
SAMPLE = datasets.SampleDataset(DATASET, _sample[DATASET.df.columns])
SAMPLE_ORG_INDICES = _sample["index"].values

In [ ]:
VALIDATION = pd.read_csv(f"{BASE_PATH}/predict_org.csv")

In [ ]:
ML_METRICS = metrics.MLClassificationMetrics(
    metrics.MLClassifier.RF,
    SAMPLE,
    seed=SEED,
    validation_df=VALIDATION,
)

In [ ]:
from k_anonymization.algorithms.clustering_based.type import ClusterAnonMethod

ALGO = KMember(
    SAMPLE,
    2,
    cluster_anon_method=ClusterAnonMethod.MEAN_MODE,
    seed=SEED,
    parallel=True,
)
ALGO_PATH = f"{BASE_PATH}/kmember"
os.makedirs(ALGO_PATH, exist_ok=True)

In [ ]:
ENCODER = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ORG_FEATURES = ENCODER.fit_transform(DATASET.df)

### Functions

In [ ]:
def get_ML_metrics_results(df):
    results = {}
    ML_METRICS.update_df(df)
    for model in metrics.MLClassifier:
        ML_METRICS.model = model.value
        ML_METRICS.validate(preview=False, restart=True)
        results[model.name] = ML_METRICS.validation_metrics
    return results

In [ ]:
def save_ml_results(results, path):
    def create_df(r):
        r_df = pd.DataFrame(r).T
        r_df.index.name = "model"
        return r_df

    create_df(results).to_csv(f"{path}/results_ml_validation.csv")

In [ ]:
def get_utility_metrics_results(df, algo):
    equivalence_qids = get_equivalence_qids(df, algo.dataset.qids_idx)
    equivalence_qids_counts = [x["count"] for x in equivalence_qids]
    results = {}
    results["DM"] = metrics.UtilityMetrics.discernibility(equivalence_qids_counts)
    results["CAVG"] = metrics.UtilityMetrics.c_avg(
        equivalence_qids_counts, algo.org_data.shape[0], algo.k
    )
    try:
        results["IL"] = metrics.UtilityMetrics.certainty_penalty_clusters_mean_mode(
            algo.clusters,
            algo.qids_idx,
            algo.is_categorical,
            algo.max_ranges,
            algo.org_data.shape[0],
        )
    except:
        results["IL"] = None

    return results

In [ ]:
def save_ut_results(results, path):
    r_df = pd.DataFrame(ut_results.items(), columns=["metric", "result"])
    r_df.to_csv(f"{path}/results_ut.csv", index=False)

In [ ]:
def attack(df, path):
    features = ENCODER.transform(df)
    cost = cdist(features, ORG_FEATURES, metric="cityblock")
    r, c = linear_sum_assignment(cost)
    d = cost[r, c]
    attack_result = pd.DataFrame({"anon_idx": r, "org_idx": c, "distance": d})
    attack_result["correct"] = attack_result["org_idx"].apply(
        lambda x: x in SAMPLE_ORG_INDICES
    )
    attack_result.to_csv(f"{path}/results_attack.csv", index=False)
    return attack_result["correct"].sum().item()

## Create Evaluation Data

### Evaluate Sample Data

In [ ]:
ml_results = get_ML_metrics_results(SAMPLE.df)

In [ ]:
save_ml_results(ml_results, f"{BASE_PATH}/sample")

In [ ]:
ut_results = get_utility_metrics_results(SAMPLE.df, ALGO)

In [ ]:
save_ut_results(ut_results, f"{BASE_PATH}/sample")

### Evaluate Anonymized Data - KMember

In [ ]:
for k in K:
    path = f"{ALGO_PATH}/k_{k}"
    os.makedirs(path, exist_ok=True)
    ALGO.k = k
    time_start = time.time()
    ALGO.anonymize()
    elapsed = time.time() - time_start
    ALGO.anon_data.to_csv(f"{path}/data.csv", index=False)
    ml_results = get_ML_metrics_results(ALGO.anon_data)
    save_ml_results(ml_results, path)
    ut_results = get_utility_metrics_results(ALGO.anon_data, ALGO)
    ut_results["RUN_TIME"] = elapsed
    save_ut_results(ut_results, path)
    attack(ALGO.anon_data, path)

### Evaluate Anonymized Data - OKA

In [ ]:
ALGO = OKA(
    SAMPLE,
    2,
    cluster_anon_method=ClusterAnonMethod.MEAN_MODE,
    seed=SEED,
    parallel=True,
)
ALGO_PATH = f"{BASE_PATH}/oka"
os.makedirs(ALGO_PATH, exist_ok=True)

In [ ]:
for k in K:
    path = f"{ALGO_PATH}/k_{k}"
    os.makedirs(path, exist_ok=True)
    ALGO.k = k
    time_start = time.time()
    ALGO.anonymize()
    elapsed = time.time() - time_start
    ALGO.anon_data.to_csv(f"{path}/data.csv", index=False)
    ml_results = get_ML_metrics_results(ALGO.anon_data)
    save_ml_results(ml_results, path)
    ut_results = get_utility_metrics_results(ALGO.anon_data, ALGO)
    ut_results["RUN_TIME"] = elapsed
    save_ut_results(ut_results, path)
    attack(ALGO.anon_data, path)

### Evaluate Anonymized Data - ClassicMondrian

In [ ]:
ALGO = ClassicMondrian(
    SAMPLE,
    2,
    cluster_anon_method=ClusterAnonMethod.MEAN_MODE,
)
ALGO_PATH = f"{BASE_PATH}/classic_mondrian"
os.makedirs(ALGO_PATH, exist_ok=True)

In [ ]:
for k in K:
    path = f"{ALGO_PATH}/k_{k}"
    os.makedirs(path, exist_ok=True)
    ALGO.k = k
    time_start = time.time()
    ALGO.anonymize()
    elapsed = time.time() - time_start
    ALGO.anon_data.to_csv(f"{path}/data.csv", index=False)
    ml_results = get_ML_metrics_results(ALGO.anon_data)
    save_ml_results(ml_results, path)
    ut_results = get_utility_metrics_results(ALGO.anon_data, ALGO)
    ut_results["RUN_TIME"] = elapsed
    save_ut_results(ut_results, path)
    attack(ALGO.anon_data, path)

### Evaluate Anonymized Data - DataFly

In [ ]:
ALGO = Datafly(
    SAMPLE,
    2,
)
ALGO_PATH = f"{BASE_PATH}/datafly"
os.makedirs(ALGO_PATH, exist_ok=True)

In [ ]:
def get_utility_metrics_results_datafly(df, algo, max_ranges):
    equivalence_qids = get_equivalence_qids(df, algo.dataset.qids_idx)
    equivalence_qids_counts = [x["count"] for x in equivalence_qids]
    results = {}
    results["DM"] = metrics.UtilityMetrics.discernibility(equivalence_qids_counts)
    results["CAVG"] = metrics.UtilityMetrics.c_avg(
        equivalence_qids_counts, algo.org_data.shape[0], algo.k
    )
    try:
        results["IL"] = metrics.UtilityMetrics.certainty_penalty_generalization(
            equivalence_qids,
            algo.org_data.shape[0],
            algo.dataset.qids_idx,
            algo.dataset.is_categorical,
            algo.dataset.hierarchies,
            max_ranges,
        )
    except:
        results["IL"] = None

    return results

In [ ]:
from k_anonymization.algorithms.clustering_based.utils import get_max_ranges

datafly_max_ranges = get_max_ranges(SAMPLE)

In [ ]:
for k in K:
    path = f"{ALGO_PATH}/k_{k}"
    os.makedirs(path, exist_ok=True)
    ALGO.k = k
    time_start = time.time()
    ALGO.anonymize()
    elapsed = time.time() - time_start
    ALGO.anon_data.to_csv(f"{path}/data.csv", index=False)
    ALGO.anon_data.age = 0  # Age is a numerical QID, and is always suppressed after Datafly anonymization in this experiment
    ml_results = get_ML_metrics_results(ALGO.anon_data)
    save_ml_results(ml_results, path)
    ut_results = get_utility_metrics_results_datafly(
        ALGO.anon_data, ALGO, datafly_max_ranges
    )
    ut_results["RUN_TIME"] = elapsed
    save_ut_results(ut_results, path)
    attack(ALGO.anon_data, path)

### Evaluate Anonymized Data - Perturbation

In [ ]:
ALGO = Perturbation(SAMPLE, 2, SEED)
ALGO_PATH = f"{BASE_PATH}/perturbation"
os.makedirs(ALGO_PATH, exist_ok=True)

In [ ]:
def get_utility_metrics_results_perturbation(df, algo):
    equivalence_qids = get_equivalence_qids(df, algo.dataset.qids_idx)
    equivalence_qids_counts = [x["count"] for x in equivalence_qids]
    results = {}
    results["DM"] = metrics.UtilityMetrics.discernibility(equivalence_qids_counts)
    results["CAVG"] = metrics.UtilityMetrics.c_avg(
        equivalence_qids_counts, algo.org_data.shape[0], algo.k
    )

    results["IL"] = (df == SAMPLE.df).sum().sum() / (
        SAMPLE.df.shape[0] * len(SAMPLE.qids)
    )

    return results

In [ ]:
for k in K:
    path = f"{ALGO_PATH}/k_{k}"
    os.makedirs(path, exist_ok=True)
    ALGO.k = k
    time_start = time.time()
    ALGO.anonymize()
    elapsed = time.time() - time_start
    ALGO.anon_data.to_csv(f"{path}/data.csv", index=False)
    ml_results = get_ML_metrics_results(ALGO.anon_data)
    save_ml_results(ml_results, path)
    ut_results = get_utility_metrics_results_perturbation(ALGO.anon_data, ALGO)
    ut_results["RUN_TIME"] = elapsed
    save_ut_results(ut_results, path)
    attack(ALGO.anon_data, path)